# Code for doing the different clustering methods, as well as data preprocessing

In this file we run the different methods

### DMACN - Deep Multi-kernel Auto-encoder Clustering Network
First load the data

In [15]:
# Example data from article about DMACN - PTSD dataset

from scipy.io import loadmat
import torch

# Load the data
PTSD = loadmat("C:\\Users\\oddar\\Downloads\\PTSD_connectivity.mat")
# PTSD is a dataset containing 87 samples (subjects) with 340 features (as vectorized functional connectivity matrices)
# The expected number of clusters are 3

# Define the functional connectivity matrix (example)
Functional_connectivity_matrix = PTSD["connectivities"]  # Example matrix

PTSD_tensor = torch.from_numpy(Functional_connectivity_matrix).float()

# Dummy data
X = PTSD_tensor  # [N,d] float tensor
print("Data shape:", X.shape)

Data shape: torch.Size([87, 340])


In [3]:
# Example data from article about DMACN - AD dataset

from scipy.io import loadmat
import torch

# Load the data
AD = loadmat("Input Data\AD_connectivity.mat")
# AD is a dataset containing 96 samples (subjects) with 888 features (as vectorized functional connectivity matrices)
# The expected number of clusters are 4

# Define the functional connectivity matrix (example)
Functional_connectivity_matrix = AD["connectivities"]  # Example matrix

AD_tensor = torch.from_numpy(Functional_connectivity_matrix).float()

# Dummy data
X = AD_tensor  # [N,d] float tensor
print("Data shape:", X.shape)

Data shape: torch.Size([96, 888])


In [1]:
# Loading the data with 29 features (for 29x29 data)
# subject_features.npz contains the vectorized upper triangle of the 29x29 FC matrices, resulting in 50 features per sample.
from scipy.io import loadmat
import torch
from data_loader import load_workable_fc


filepath = "Prosjektoppgave-Odd-Arne-og-Mats-main\\subject_features.npz" # All subjects, 50 features

Functional_connectivity_matrix = load_workable_fc(filepath)

Functional_connectivity_matrix.to_csv("subject_features_clean.csv", index=True)

print(Functional_connectivity_matrix.shape)

X = torch.from_numpy(Functional_connectivity_matrix.values).float()


(141, 50)
(141, 29)
(141, 29)


In [ ]:
# Loading the data with all 200 features (for 200x200 data)
# 200_schaefer_vectorized_fc.mat contains the vectorized upper triangle of the 200x200 FC matrices, resulting in 19900 features per sample.

from scipy.io import loadmat
import torch
import numpy as np

FC_test_mat = loadmat("C:\\Users\\oddar\\Downloads\\200_schaefer_vectorized_fc.mat")
# FC_test_mat = loadmat("C:\\Mats og Odd Arne\\Prosjektoppgave\\sch407\\YA\\200_schaefer_vectorized_fc.mat")  # Load the .mat file

FC_test_array = FC_test_mat["200_vectorized_fc"]  # Example matrix
# np.fill_diagonal(FC_test_array, 1.0)  # Set diagonal to zero

print(FC_test_array[0:5].shape)  # Print the first 5 rows to verify

X = torch.from_numpy(FC_test_array).float()  # Example matrix

# Check if any values are abs(X) > 1.0
if torch.any(torch.abs(X) > 1.0):
    print("Warning: Some values in X have absolute value greater than 1.0, which may cause numerical issues in the polynomial kernel.")

(5, 19900)


In [4]:
#  Loading the data with all 200 features (for 200x200 data) - alternative loading method

from scipy.io import loadmat
import torch
import numpy as np

filepath = "Input Data\FC1.mat"
fc_mat = loadmat(filepath)
fc_mat = fc_mat['FC1']  # Extract the FC1 variable from the loaded .mat file

print(fc_mat.shape)  # Should print (200, 200, N_subjects)

fc_mat_inv = np.transpose(fc_mat, (2, 0, 1))  # Transpose to (N_subjects, 200, 200)
fc_mat_inv = fc_mat_inv[:, None, :, :]  # Add a channel dimension to get (N_subjects, 1, 200, 200)
print(fc_mat_inv.shape)  # Should print (N_subjects, 1, 200, 200)

X = torch.from_numpy(fc_mat_inv).float()  # Convert to PyTorch tensor
print("Data shape:", X.shape)  # Should print (N_subjects, 1, 200, 200)

# Expected shape is (Batch size, channels, height, width)

(200, 200, 72)
(72, 1, 200, 200)
Data shape: torch.Size([72, 1, 200, 200])


In [1]:
from data_loader import load_static_functional_connectiviies

ADHD = load_static_functional_connectiviies("Input Data\ADHD_connectivity.mat")

Original FC shape: (487, 672)
Final FC shape: (487, 377)


Define the autoencoder specs

In [13]:
kernel_specs = [
    {"kind": "rbf", "t0": 0.01},
    {"kind": "rbf", "t0": 0.05},
    {"kind": "rbf", "t0": 0.1},
    {"kind": "rbf", "t0": 1},
    {"kind": "rbf", "t0": 10},
    {"kind": "rbf", "t0": 50},
    {"kind": "rbf", "t0": 100},
    {"kind": "poly", "a": 0, "b": 2},
    {"kind": "poly", "a": 0, "b": 4},
    {"kind": "poly", "a": 1, "b": 2},
    {"kind": "poly", "a": 1, "b": 4}
]  # h = 3

In [17]:
# Config for 340x340 data PTSD
from DMACN import DMACN, DMACNConfig


cfg = DMACNConfig(
    name="PTSD_340x340",
    C=3,  # number of clusters
    dims_enc=[340, 285, 240, 202, 170],   # mid = 2 encoder Linear layers = L/2
    dims_dec=[170, 202, 240, 285, 340],
    kernel_specs=kernel_specs,
    m_fuzz=1.08,
    lam1=0.5,
    lam2=0.5,
    lr=1e-3,
    epochs=200,
    mk_max_iters=20,
    mk_eps_stop=1e-5,
    renormalize_omega_sum1=True,
    mid_only_first=True,
    mid_only_last=True,
)

In [1]:
# Config for 888x888 data AD
from DMACN import DMACN, DMACNConfig


cfg = DMACNConfig(
    name="AD_888x888",
    C=4,  # number of clusters
    dims_enc=[888, 747, 628, 528, 444],   # mid = 2 encoder Linear layers = L/2
    dims_dec=[444, 528, 628, 747, 888],
    kernel_specs=kernel_specs,
    m_fuzz=1.08,
    lam1=1e-2,
    lam2=0.5,
    lr=1e-3,
    epochs=500,
    mk_max_iters=20,
    mk_eps_stop=1e-5,
    renormalize_omega_sum1=True,
    mid_only_first=True,
    mid_only_last=True,
)

KeyboardInterrupt: 

In [35]:
from DMACN import DMACN, DMACNConfig


# Config for 29x29 data
cfg = DMACNConfig(
    name = "DMACN_29x29",
    C=3,  # number of clusters
    dims_enc=[29, 25, 22, 19, 17],   # N / 2^(i/l). l=number of layers, N=number of features 
    dims_dec=[17, 19, 22, 25, 29],
    kernel_specs=kernel_specs,
    m_fuzz=1.08,
    lam1=0.5,
    lam2=0.5,
    lr=1e-3,
    epochs=500,
    mk_max_iters=20,
    mk_eps_stop=1e-5,
    renormalize_omega_sum1=True,
    mid_only_first=True,
    mid_only_last=True,
)


In [ ]:
from DMACN import DMACN, DMACNConfig

# Config for 19900x19900 data
cfg = DMACNConfig(
    C=3,  # number of clusters
    dims_enc=[19900, 15795, 12536, 9950],   # N / 2^(i/l). l=number of layers, N=number of features 
    dims_dec=[9950, 12536, 15795, 19900],
    kernel_specs=kernel_specs,
    m_fuzz=1.08,
    lam1=100,
    lam2=0.5,
    lr=1e-3,
    epochs=200,
    mk_max_iters=200,
    mk_eps_stop=1e-5,
    renormalize_omega_sum1=True,
    mid_only_first=True,
    mid_only_last=True,
)


Run the model

In [25]:
model = DMACN(cfg)
model.fit(X, verbose_every=50)
labels = model.predict(save=True)
print("labels shape:", labels.shape)
print("labels: ", labels)

Ready to train DMACN:
epoch    1/500 [mid-only]  J=2.4553e+03  J1=2.0265e+03  J2=5.8576e-07  J3=4.2879e+02  omega_sum=1.0000
epoch   50/500 [multilayer]  J=2.2547e+03  J1=1.8458e+03  J2=1.4109e-07  J3=4.0886e+02  omega_sum=1.0000
epoch  100/500 [multilayer]  J=2.2347e+03  J1=1.8458e+03  J2=1.3963e-07  J3=3.8897e+02  omega_sum=1.0000
epoch  150/500 [multilayer]  J=2.2158e+03  J1=1.8457e+03  J2=1.3440e-07  J3=3.7004e+02  omega_sum=1.0000
epoch  200/500 [multilayer]  J=2.1977e+03  J1=1.8457e+03  J2=1.2529e-07  J3=3.5205e+02  omega_sum=1.0000
epoch  250/500 [multilayer]  J=2.1805e+03  J1=1.8456e+03  J2=1.0800e-07  J3=3.3495e+02  omega_sum=1.0000
epoch  300/500 [multilayer]  J=2.1641e+03  J1=1.8454e+03  J2=7.7536e-08  J3=3.1870e+02  omega_sum=1.0000
epoch  350/500 [multilayer]  J=2.1417e+03  J1=1.8384e+03  J2=4.1649e-08  J3=3.0331e+02  omega_sum=1.0000
epoch  400/500 [multilayer]  J=2.1025e+03  J1=1.8133e+03  J2=1.6338e-08  J3=2.8918e+02  omega_sum=1.0000
X: tensor([[nan, nan, nan,  ..., na

ValueError: Gaussian kernel K contains Inf or NaN values. Check for numerical issues.

In [ ]:
# Evaluate PTSD clustering performance using PTSD_clinical_labels.scv.csv
import pandas as pd
clinical_labels = pd.read_csv("PTSD_clinical.scv.csv")
clinical_labels = clinical_labels.iloc[:, 0]  # Assuming the first column contains the labels
true_labels = [0] + clinical_labels.tolist()  # Add a 0 at the beginning to match the number of samples (87)

# Evaluate clustering performance using Adjusted Rand Index (ARI)
from sklearn.metrics import adjusted_rand_score
ari = adjusted_rand_score(true_labels, labels)
print("Adjusted Rand Index (ARI):", ari)

Adjusted Rand Index (ARI): 0.9747477598769758


In [56]:
# Evaluate AD clustering performance using AD_clinical.csv
import pandas as pd
clinical_labels = pd.read_csv("AD_clinical.csv")
clinical_labels = clinical_labels.iloc[:, 0]  # Assuming the first column contains the labels
true_labels = [0] + clinical_labels.tolist()  # Add a 0 at the beginning to match the number of samples (87)

# Evaluate clustering performance using Adjusted Rand Index (ARI)
from sklearn.metrics import adjusted_rand_score
ari = adjusted_rand_score(true_labels, labels)
print("Adjusted Rand Index (ARI):", ari)

Adjusted Rand Index (ARI): 0.08397546974963356


### CAE

In [ ]:
from Convolutional_AE import DCEC, pretrain_cae, initialize_cluster_centers, train_dcec, predict_soft_assignments
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(X)
dataloader = DataLoader(dataset, batch_size=16, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# TODO: Add the ability to save labels and the model
# TODO: run the model for multiple cluster numbers and evaluate the clustering performance using Calinski-Harabasz index and silhouette score

model = DCEC(latent_dim=16, n_clusters=4, alpha=1.0).to(device)

pretrain_cae(model, dataloader, device)
y_pred_initial = initialize_cluster_centers(model, dataloader, device)
print("Early cluster centers initialized.", y_pred_initial)

train_dcec(
    model,
    dataloader,
    device,
    gamma=0.01,
    epochs=100,
    lr=1e-3,
    update_interval=10)

q_final = predict_soft_assignments(model, dataloader, device)
labels = q_final.argmax(axis=1)
print("Predicted cluster labels:", labels)

[Pretrain] Epoch 001/50 - Recon loss: 0.086121
[Pretrain] Epoch 010/50 - Recon loss: 0.044449
[Pretrain] Epoch 020/50 - Recon loss: 0.037014
[Pretrain] Epoch 030/50 - Recon loss: 0.036189
[Pretrain] Epoch 040/50 - Recon loss: 0.035821
[Pretrain] Epoch 050/50 - Recon loss: 0.035461
Early cluster centers initialized. [3 0 3 1 2 2 0 1 0 2 0 0 3 3 0 0 2 1 0 0 2 0 2 2 0 0 1 0 1 0 3 0 1 0 3 0 2
 2 2 2 2 0 3 0 1 3 0 1 0 1 3 1 3 1 1 0 3 3 0 0 3 3 3 2 2 1 3 1 1 1 0 3]
[DCEC] Epoch 000/100 - label change fraction: 0.000000
[DCEC] Epoch 001/100 - Total: 0.055933, Recon: 0.054648, KL: 0.128487
[DCEC] Epoch 010/100 - label change fraction: 0.236111
[DCEC] Epoch 020/100 - Total: 0.035677, Recon: 0.035080, KL: 0.059680
[DCEC] Epoch 020/100 - label change fraction: 0.069444
[DCEC] Epoch 030/100 - label change fraction: 0.041667
[DCEC] Epoch 040/100 - Total: 0.035248, Recon: 0.034449, KL: 0.079869
[DCEC] Epoch 040/100 - label change fraction: 0.013889
[DCEC] Epoch 050/100 - label change fraction: 0.000

In [6]:
z = model.cae.encode(X.to(device)).cpu().detach().numpy()
print("Embeddings shape:", z.shape)
print(np.std(z, axis=0))

Embeddings shape: (72, 16)
[0.46560612 0.4006527  0.4108344  0.40220624 0.30976105 0.3334723
 0.386272   0.36025178 0.45331877 0.46854222 0.33685458 0.43235037
 0.43731597 0.62509197 0.91978854 0.357025  ]


In [21]:
# Evaluate AD clustering performance using AD_clinical.csv
from Evaluate_models import evaluate_single_clustering

triu_idx = np.triu_indices(200, k=1)
fc_mat_transp = np.transpose(fc_mat, (2, 0, 1))  # Transpose to (N_subjects, 200, 200)

Features = np.array([
    fc_mat_transp[i][triu_idx] for i in range(fc_mat_transp.shape[0])
]) 
predictions = labels

evaluate_single_clustering(Features, predictions)



 Scores for the given labels:
Silhouette coefficient: 0.0007629846376234093
Davies-Bouldin score: 4.903744607940154
Calinski-Harabasz score: 1.6387715400984655


### UMAP

In [ ]:
from UMAP import UMAP

C:\Users\oddar\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\oddar\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


ImportError: cannot import name 'UMAP' from 'UMAP' (c:\Users\oddar\OneDrive - NTNU\Documents\Universitet\Masteroppgåve\Master-Thesis-Autoencoder-and-UMAP-clustering-on-functional-connectivity\UMAP.py)

### HDBSCAN
Perform HDBSCAN on the data

In [ ]:
from HDBSCAN import hdbscan_clustering

hdbscan_clustering(Functional_connectivity_matrix=Functional_connectivity_matrix, save_labels=True)

### Evaluate the clusters
Evaluate the clusters using simple methods: Silhouette coefficient, Davies-Bouldin score and Calinski-Harabasz score

In [6]:
from Evaluate_models import evaluate_clustering, evaluate_single_clustering
import os

# Define where to find the labels 
labels_path = "Clusters\DMACN__Clusters_3__label_0_21_label_1_19_label_2_32.txt"
evaluate_single_clustering(functional_connectivity_matrix=X, labels_path=labels_path)


 Scores for the given labels:
Silhouette coefficient: 0.002338581718504429
Davies-Bouldin score: 5.71068601622989
Calinski-Harabasz score: 1.7393631346468488
